# Esteira de Estoque · 3 Relatórios de Insight (D-1 / D-2 / D-3)

**Escopo pequeno e fechado.** Esta esteira:
1. **Carrega o insumo** `retail_store_inventory.csv` (mesmo schema do dataset do Kaggle).
2. **Enriquece** o dado (receita, ruptura, venda perdida, estoque mínimo, ponto de pedido).
3. **Roda a esteira** simulando **Lambda → Glue → Step Functions**.
4. Emite **3 relatórios Excel**, um por insight:

| Relatório | Insight de negócio | Pergunta |
|-----------|--------------------|----------|
| **D-1** | Produtos vendidos | *O que mais saiu e onde se concentra a receita?* |
| **D-2** | Reposição necessária | *O que está em ruptura e precisa repor?* |
| **D-3** | Tendência de consumo | *O consumo está subindo ou caindo?* |

> Cada serviço da AWS é simulado como uma peça Python curta. No fim, os 3 `.xlsx` saem prontos com fórmulas e um destaque de insight.


## 0 · Setup

In [2]:
import time, random
from pathlib import Path
from datetime import date, timedelta, datetime
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

random.seed(11); np.random.seed(11)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 130)
OUT = Path(".")
print("setup ok")

setup ok


## 1 · Insumo (`retail_store_inventory.csv`)

Usa o CSV na raiz do projeto se existir (Kaggle ou arquivo real). Só gera amostra sintética se o arquivo não estiver presente.

In [13]:
# ============================================================================
# CÉLULA DE INGESTÃO DO INSUMO
# Regra: se o CSV real existir no caminho, carrega; senão, gera um sintético
# de mesmo schema (fallback) para a esteira rodar de ponta a ponta.
# ============================================================================
INSUMO_PATH = OUT / "retail_store_inventory.csv"

# Contrato de schema: as 15 colunas que o resto da esteira espera encontrar.
SCHEMA = [
    "Date", "Store ID", "Product ID", "Category", "Region",
    "Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast",
    "Price", "Discount", "Weather Condition", "Holiday/Promotion",
    "Competitor Pricing", "Seasonality",
]

if INSUMO_PATH.exists():
    # ---- CAMINHO 1: CSV real (ex.: baixado do Kaggle) ----
    insumo = pd.read_csv(INSUMO_PATH, parse_dates=["Date"])
    # Gate de contrato: aborta cedo se faltar coluna (evita erro lá na frente).
    missing = [c for c in SCHEMA if c not in insumo.columns]
    if missing:
        raise ValueError(f"CSV incompleto — colunas faltando: {missing}")
    print(f"insumo carregado: {INSUMO_PATH.resolve()}")
else:
    # ---- CAMINHO 2: fallback sintético (mesmo schema do Kaggle) ----
    STORES = ["S001", "S002"]
    PRODUCTS = {"P01": "Groceries", "P02": "Groceries", "P03": "Electronics",
                "P04": "Clothing",  "P05": "Toys"}
    REGION = {"S001": "South", "S002": "North"}
    BASE = {"P01": 90, "P02": 70, "P03": 30, "P04": 45, "P05": 55}
    DAYS = [date(2026, 6, 15) + timedelta(days=i) for i in range(7)]

    rows = []
    for d in DAYS:
        weekend = 1.3 if d.weekday() >= 5 else 1.0          # fim de semana puxa demanda
        holiday = int(random.random() < 0.15)               # evento/promoção esporádico
        for s in STORES:
            for p, cat in PRODUCTS.items():
                demand = BASE[p] * weekend * (1.4 if holiday else 1.0) * np.random.uniform(0.8, 1.2)
                inv = max(0, int(np.random.normal(demand * 1.05, demand * 0.45)))
                sold = int(min(inv, round(demand)))         # venda é limitada pelo estoque -> gera ruptura
                price = round({"Groceries": 8, "Electronics": 120, "Clothing": 45, "Toys": 30}[cat]
                              * np.random.uniform(0.9, 1.1), 2)
                rows.append({
                    "Date": d.isoformat(), "Store ID": s, "Product ID": p, "Category": cat,
                    "Region": REGION[s], "Inventory Level": inv, "Units Sold": sold,
                    "Units Ordered": int(max(0, np.random.normal(demand * 0.9, demand * 0.2))),
                    "Demand Forecast": round(demand * np.random.uniform(0.9, 1.1), 1),
                    "Price": price, "Discount": int(np.random.choice([0, 5, 10], p=[0.7, 0.2, 0.1])),
                    "Weather Condition": np.random.choice(["Sunny", "Rainy", "Cloudy"]),
                    "Holiday/Promotion": holiday,
                    "Competitor Pricing": round(price * np.random.uniform(0.9, 1.1), 2),
                    "Seasonality": "Summer",
                })
    insumo = pd.DataFrame(rows)
    insumo.to_csv(INSUMO_PATH, index=False)
    print(f"insumo sintético criado (fallback): {INSUMO_PATH.resolve()}")

# ============================================================================
# PRINCIPAIS ANÁLISES DO INSUMO (sanidade antes de enriquecer)
# ============================================================================
insumo["Date"] = pd.to_datetime(insumo["Date"])

# 1) Volume e granularidade — confirma o grão esperado (1 linha por Data×Loja×Produto)
print(f"\n[1] Volume: {insumo.shape[0]:,} linhas x {insumo.shape[1]} colunas")
print(f"    Período: {insumo['Date'].min().date()} → {insumo['Date'].max().date()} "
      f"({insumo['Date'].nunique()} dias)")
print(f"    Lojas: {insumo['Store ID'].nunique()} | "
      f"Produtos: {insumo['Product ID'].nunique()} | "
      f"Categorias: {insumo['Category'].nunique()}")
dups = insumo.duplicated(["Date", "Store ID", "Product ID"]).sum()
print(f"    Duplicatas na chave (Data,Loja,Produto): {dups}")

# 2) Qualidade — nulos e negativos nas colunas numéricas (base do Bronze→Silver)
num_cols = ["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast", "Price"]
print(f"\n[2] Nulos totais: {int(insumo.isna().sum().sum())} | "
      f"Negativos: {int((insumo[num_cols] < 0).sum().sum())}")

# 3) Concentração de vendas — onde está o volume (prévia do insight D-1)
top_cat = insumo.groupby("Category")["Units Sold"].sum().sort_values(ascending=False)
print(f"\n[3] Vendas por categoria (un.):\n{top_cat.to_string()}")

# 4) Indício de ruptura — % de linhas que venderam todo o estoque com demanda acima
#    (prévia do insight D-2; é o sinal de "vendeu tudo, venderia mais")
ruptura = ((insumo["Units Sold"] >= insumo["Inventory Level"]) &
           (insumo["Demand Forecast"] > insumo["Inventory Level"])).mean() * 100
print(f"\n[4] Indício de ruptura: {ruptura:.1f}% das linhas")

insumo.head()

insumo carregado: C:\welligton-aws\project-datamesh-1\retail_store_inventory.csv

[1] Volume: 73,100 linhas x 15 colunas
    Período: 2022-01-01 → 2024-01-01 (731 dias)
    Lojas: 5 | Produtos: 20 | Categorias: 5
    Duplicatas na chave (Data,Loja,Produto): 0

[2] Nulos totais: 0 | Negativos: 673

[3] Vendas por categoria (un.):
Category
Furniture      2025017
Groceries      2000482
Clothing       1999166
Toys           1990485
Electronics    1960432

[4] Indício de ruptura: 0.3% das linhas


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [21]:
import shutil  # necessário para sobrescrever partições (idempotência)
# obs: a escrita usa parquet -> precisa de pyarrow (pip install pyarrow)

# ============================================================================
# CARGA INCREMENTAL POR PARTIÇÃO (esteira diária)
# A "tabela" é uma PASTA particionada por dia. Cada dia = 1 partição dt=AAAA-MM-DD.
# Incrementar = gravar mais uma partição. Reprocessar um dia = sobrescreve só ela.
# ============================================================================
ORIGEM_DIR = OUT / "tabela_origem"        # tabela de origem (1 partição por dia)
ENRIQ_DIR  = OUT / "tabela_enriquecida"   # tabela enriquecida (incremental)
ORIGEM_DIR.mkdir(exist_ok=True)
ENRIQ_DIR.mkdir(exist_ok=True)

insumo["Date"] = pd.to_datetime(insumo["Date"])

def _part(base, dt):
    return Path(base) / f"dt={pd.Timestamp(dt).date()}"

def carregar_origem_dia(dt):
    """Extrai as linhas de UM dia do insumo e grava como partição de origem.
    Idempotente: regravar o mesmo dia sobrescreve só aquela partição."""
    dt = pd.Timestamp(dt).normalize()
    dia = insumo[insumo["Date"].dt.normalize() == dt].copy()
    if dia.empty:
        raise ValueError(f"Sem dados para {dt.date()} no insumo.")
    part = _part(ORIGEM_DIR, dt)
    if part.exists():
        shutil.rmtree(part)                # overwrite da partição -> não duplica
    part.mkdir(parents=True)
    dia.to_parquet(part / "data.parquet", index=False)
    return dia

def enriquecer_dia(dt):
    """Lê a origem do dia, aplica enriquecimento de LINHA e grava como
    partição da tabela enriquecida (incremental + idempotente)."""
    dt = pd.Timestamp(dt).normalize()
    dia = pd.read_parquet(_part(ORIGEM_DIR, dt) / "data.parquet")
    dia["_revenue"]    = (dia["Units Sold"] * dia["Price"]).round(2)
    dia["_stockout"]   = ((dia["Units Sold"] >= dia["Inventory Level"]) &
                          (dia["Demand Forecast"] > dia["Inventory Level"])).astype(int)
    dia["_lost"]       = np.where(dia["_stockout"] == 1,
                                  (dia["Demand Forecast"] - dia["Units Sold"]).clip(lower=0), 0).round(1)
    dia["_is_weekend"] = (pd.to_datetime(dia["Date"]).dt.weekday >= 5).astype(int)
    dia["dt"]          = str(dt.date())    # coluna de partição
    part = _part(ENRIQ_DIR, dt)
    if part.exists():
        shutil.rmtree(part)
    part.mkdir(parents=True)
    dia.to_parquet(part / "data.parquet", index=False)
    return dia

def ler_enriquecido():
    """Lê a tabela enriquecida COMPLETA (todas as partições já processadas)."""
    files = sorted(Path(ENRIQ_DIR).glob("dt=*/data.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

def processar_dia(dt):
    """Esteira de UM dia: origem -> enriquecido. Retorna a tabela acumulada."""
    carregar_origem_dia(dt)
    enriquecer_dia(dt)
    acc = ler_enriquecido()
    n_part = len(list(Path(ENRIQ_DIR).glob("dt=*")))
    print(f"  processado {pd.Timestamp(dt).date()} | "
          f"partições: {n_part} | linhas acumuladas: {len(acc):,}")
    return acc

# ----------------------------------------------------------------------------
# SIMULAÇÃO INCREMENTAL: 1 a 3 dias
# Hoje processamos só o dia 01. Dias 02 e 03 ficam prontos, é só descomentar
# quando quiser incrementar a tabela.
# ----------------------------------------------------------------------------
dias = sorted(insumo["Date"].dt.normalize().unique())
print("período disponível:", pd.Timestamp(dias[0]).date(), "→", pd.Timestamp(dias[-1]).date(),
      f"({len(dias)} dias)\n")

print("[1] processa o dia 01:")
acc = processar_dia(dias[0])

print("\n[2] adiciona o dia 02 -> a tabela incrementa:")
acc = processar_dia(dias[1])

print("\n[3] adiciona o dia 03 -> incrementa de novo:")
acc = processar_dia(dias[2])

acc.tail()

período disponível: 2022-01-01 → 2024-01-01 (731 dias)

[1] processa o dia 01:
  processado 2022-01-01 | partições: 2 | linhas acumuladas: 200

[2] adiciona o dia 02 -> a tabela incrementa:
  processado 2022-01-02 | partições: 2 | linhas acumuladas: 200

[3] adiciona o dia 03 -> incrementa de novo:
  processado 2022-01-03 | partições: 3 | linhas acumuladas: 300


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,_revenue,_stockout,_lost,_is_weekend,dt
295,2022-01-03,S005,P0016,Clothing,North,361,17,171,25.61,80.31,5,Cloudy,1,76.56,Spring,1365.27,0,0.0,0,2022-01-03
296,2022-01-03,S005,P0017,Toys,East,154,19,116,37.45,41.28,10,Rainy,0,40.59,Spring,784.32,0,0.0,0,2022-01-03
297,2022-01-03,S005,P0018,Toys,North,376,36,53,43.47,38.42,0,Snowy,1,35.98,Winter,1383.12,0,0.0,0,2022-01-03
298,2022-01-03,S005,P0019,Toys,North,268,207,177,198.44,59.93,20,Snowy,0,64.67,Autumn,12405.51,0,0.0,0,2022-01-03
299,2022-01-03,S005,P0020,Toys,South,451,91,193,99.88,66.76,5,Sunny,0,62.53,Summer,6075.16,0,0.0,0,2022-01-03


In [23]:
# ============================================================================
# RELATÓRIO D-1 · PRODUTOS VENDIDOS
# Executado no dia 02, mas processa o dado de D-1 (o dia anterior).
# ============================================================================
from openpyxl import Workbook

DATA_EXECUCAO = pd.Timestamp("2022-01-02").normalize()        # quando a esteira roda ("hoje")
DIA_DADO      = (DATA_EXECUCAO - pd.Timedelta(days=1)).normalize()  # D-1 = o dado processado

print(f"execução: {DATA_EXECUCAO.date()}  |  processando dado D-1: {DIA_DADO.date()}\n")

part_dia = _part(ENRIQ_DIR, DIA_DADO) / "data.parquet"

# trava: se o D-1 ainda não foi enriquecido, processa agora (idempotente)
if not part_dia.exists():
    print(f"partição dt={DIA_DADO.date()} ausente — processando...")
    processar_dia(DIA_DADO)

# lê SÓ a partição do D-1
enr_dia = pd.read_parquet(part_dia)

# agrega por produto (soma as lojas) -> grão do relatório D-1
agg = (enr_dia.groupby(["Product ID", "Category"])
              .agg(unidades=("Units Sold", "sum"),
                   receita=("_revenue", "sum"))
              .reset_index()
              .sort_values("unidades", ascending=False))

total_u = int(agg["unidades"].sum())
top = agg.iloc[0]
top3 = agg.head(3)["unidades"].sum() / total_u * 100

# ---- monta o Excel ----
wb = Workbook(); ws = wb.active; ws.title = "D1_Produtos_Vendidos"
BLUE = "2563EB"; LT = "DBEAFE"

_title(ws, "Relatório D-1 · Produtos Vendidos",
       f"Execução {DATA_EXECUCAO.date()} | dado D-1: {DIA_DADO.date()} | gerado pela esteira",
       5, BLUE)
_insight(ws, f"No dado de {DIA_DADO.date()} (D-1), o produto líder foi "
             f"{top['Product ID']} ({top['Category']}) com {int(top['unidades'])} un. "
             f"Os 3 primeiros concentram {top3:.0f}% das vendas.", 5, LT)
_header(ws, 6, ["Produto", "Categoria", "Unid. Vendidas", "Receita (R$)", "% do Total"], BLUE)

r0 = 7
for i, row in enumerate(agg.itertuples(index=False)):
    r = r0 + i
    _cell(ws, r, 1, row[0])
    _cell(ws, r, 2, row[1], align="left")
    _cell(ws, r, 3, int(row.unidades), fmt="#,##0")
    _cell(ws, r, 4, float(row.receita), fmt="#,##0.00")
    _cell(ws, r, 5, f"=C{r}/$C${r0+len(agg)}", fmt="0.0%")     # % via fórmula

tr = r0 + len(agg)   # linha de TOTAL
_cell(ws, tr, 1, "TOTAL", bold=True, align="left"); _cell(ws, tr, 2, "")
_cell(ws, tr, 3, f"=SUM(C{r0}:C{tr-1})", fmt="#,##0", bold=True)
_cell(ws, tr, 4, f"=SUM(D{r0}:D{tr-1})", fmt="#,##0.00", bold=True)
_cell(ws, tr, 5, f"=SUM(E{r0}:E{tr-1})", fmt="0.0%", bold=True)

for col, w in zip("ABCDE", [12, 14, 16, 16, 12]):
    ws.column_dimensions[col].width = w

# nome do arquivo carrega as DUAS datas: quando rodou e qual dado fechou
REL_PATH = OUT / f"relatorio_D1_exec{DATA_EXECUCAO.date()}_dado{DIA_DADO.date()}.xlsx"
wb.save(REL_PATH)
print(f"\nrelatório gerado: {REL_PATH.resolve()}")
print(f"  dado {DIA_DADO.date()} | {len(agg)} produtos | {total_u:,} un. vendidas")
agg

execução: 2022-01-02  |  processando dado D-1: 2022-01-01


relatório gerado: C:\welligton-aws\project-datamesh-1\relatorio_D1_exec2022-01-02_dado2022-01-01.xlsx
  dado 2022-01-01 | 69 produtos | 14,484 un. vendidas


C:\Users\cloud\AppData\Local\Temp\ipykernel_16964\4288848922.py:8: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  DIA_DADO      = (DATA_EXECUCAO - pd.Timedelta(days=1)).normalize()  # D-1 = o dado processado


,Product ID,Category,unidades,receita
47,P0014,Electronics,962,73436.83
48,P0015,Clothing,810,52478.25
2,P0002,Clothing,619,30942.66
63,P0019,Clothing,506,25483.75
1,P0001,Toys,495,19528.30
...,...,...,...,...
51,P0016,Clothing,11,179.30
28,P0009,Clothing,9,731.61
32,P0010,Clothing,8,296.32
3,P0002,Electronics,4,112.80
